# Left-wing vote share per municipality

This notebook calculates the share of left-wing votes per municipality in the Tweede Kamer election of 29 October 2025, maps it, and saves the result for the analysis.

**Definition of left-wing** (the same as in notebook 6):
- **Broad (main definition):** GroenLinks-PvdA, SP, Partij voor de Dieren, DENK, BIJ1, D66 and Volt
- **Narrow (robustness check):** the same without D66 and Volt

**Required files:**
- `data/raw/tk2025_uitslag.csv`: official election results per municipality (Kiesraad, via data.overheid.nl)
- `data/raw/gemeenten_2024.gpkg`: municipal boundaries from CBS/PDOK, 2024 municipal division
- `data/processed/chargingpoints_households_income.csv`: created by notebook 1, only used to check that the names match

**Output:**
- `data/processed/left_wing_votes_per_municipality.csv`
- `figures/map_left_wing_votes_per_municipality.png`

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 1. Load the election results

The file is in long format: one row per region, party and type of number. We keep only:
- rows for municipalities (`RegioCode` starts with `G`),
- the number of votes per party list (`VeldType` = `LijstAantalStemmen`).

The region `NBSB` contains postal votes from Dutch citizens abroad. It is not a municipality and is removed.

In [ ]:
election = pd.read_csv("../data/raw/tk2025_uitslag.csv", sep=";", dtype=str)

votes = election[
    election["RegioCode"].str.startswith("G")
    & (election["VeldType"] == "LijstAantalStemmen")
].copy()
votes = votes[votes["Regio"] != "NBSB"]
votes["Waarde"] = votes["Waarde"].astype(int)

print("Number of municipalities:", votes["Regio"].nunique())
display(votes[["Regio", "LijstNaam", "Waarde"]].head())

## 2. Calculate the left-wing vote share

We make a table with one row per municipality and one column per party. The total number of valid votes per municipality is the sum over all parties.

In [ ]:
per_party = votes.pivot_table(index="Regio", columns="LijstNaam", values="Waarde", aggfunc="sum").fillna(0)
total_votes = per_party.sum(axis=1)

LEFT_NARROW = [
    "GROENLINKS / Partij van de Arbeid (PvdA)",
    "SP (Socialistische Partij)",
    "Partij voor de Dieren",
    "DENK",
    "BIJ1",
]
LEFT_BROAD = LEFT_NARROW + ["D66", "Volt"]

left = pd.DataFrame({
    "municipality": per_party.index.str.strip(),
    "left_wing_vote_pct": (per_party[LEFT_BROAD].sum(axis=1) / total_votes * 100).round(2).values,
    "left_wing_vote_narrow_pct": (per_party[LEFT_NARROW].sum(axis=1) / total_votes * 100).round(2).values,
})

print("Number of municipalities:", len(left))
display(left.describe().round(1))

## 3. Harmonise municipality names

The Kiesraad writes some names differently from the charging point file, for example *Hengelo (O)* instead of *Hengelo*. We use the names from the charging point file everywhere, so all processed files can be merged on `municipality`.

In [ ]:
name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Hengelo (O)": "Hengelo",
    "Bergen (L)": "Bergen (L.)",
    "Bergen (NH)": "Bergen (NH.)",
}
left["municipality"] = left["municipality"].replace(name_mapping)

# Check: does every municipality match the charging point data?
charging = pd.read_csv("../data/processed/chargingpoints_households_income.csv")
print("Only in the election data:", set(left["municipality"]) - set(charging["municipality"]))
print("Only in the charging data:", set(charging["municipality"]) - set(left["municipality"]))

## 4. Load municipal boundaries and join the data

The boundary file uses CBS names. We convert these to the same names as the charging point file (the same mapping as in notebook 2).

In [ ]:
municipalities = gpd.read_file("../data/raw/gemeenten_2024.gpkg", layer="gemeente_gegeneraliseerd")

map_name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein",
}
municipalities["municipality"] = municipalities["statnaam"].replace(map_name_mapping)

map_gdf = municipalities.merge(left, on="municipality", how="left")

# Check: both lines should print set() and 0
print("Only on the map:", set(municipalities["municipality"]) - set(left["municipality"]))
print("Municipalities without data:", map_gdf["left_wing_vote_pct"].isna().sum())

## 5. Create the maps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

map_gdf.plot(
    column="left_wing_vote_pct",
    cmap="Reds",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Left-wing votes (% of valid votes)", "shrink": 0.6},
    ax=axes[0],
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
axes[0].set_title("Left-wing vote share, broad definition\n(incl. D66 and Volt), 2025", fontsize=14)
axes[0].axis("off")

map_gdf.plot(
    column="left_wing_vote_narrow_pct",
    cmap="Reds",
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Left-wing votes (% of valid votes)", "shrink": 0.6},
    ax=axes[1],
    missing_kwds={"color": "lightgrey", "label": "No data"},
)
axes[1].set_title("Left-wing vote share, narrow definition\n(excl. D66 and Volt), 2025", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/map_left_wing_votes_per_municipality.png", dpi=200)
plt.show()

## 6. Save for the analysis

In [ ]:
left.to_csv("../data/processed/left_wing_votes_per_municipality.csv", index=False)
print("Saved:", len(left), "municipalities")